In [1]:
from collections import defaultdict
from itertools import combinations

In [2]:
def eclat(prefix, items, min_support, freq_itemsets):
    while items:
        i, itids = items.pop()
        isupp = len(itids)
        if isupp >= min_support:
            itemset = prefix + [i]
            freq_itemsets[frozenset(itemset)] = isupp
            suffix = []
            for j, jtids in items:
                inter = itids & jtids
                if len(inter) >= min_support:
                    suffix.append((j, inter))
            eclat(prefix + [i], sorted(suffix, key=lambda x: len(x[1]), reverse=True), min_support, freq_itemsets)
    return freq_itemsets

In [3]:
def generate_rules(freq_itemsets, total_transactions, min_confidence=0.5):
    rules = []
    for itemset, support_count in freq_itemsets.items():
        if len(itemset) > 1:  # rules only for itemsets with 2+
            for i in range(1, len(itemset)):
                for antecedent in combinations(itemset, i):
                    antecedent = frozenset(antecedent)
                    consequent = itemset - antecedent
                    support = support_count / total_transactions
                    confidence = support_count / freq_itemsets[antecedent]
                    lift = confidence / (freq_itemsets[consequent] / total_transactions)

                    if confidence >= min_confidence:
                        rules.append({
                            'antecedent': list(antecedent),
                            'consequent': list(consequent),
                            'support': round(support, 3),
                            'confidence': round(confidence, 3),
                            'lift': round(lift, 3)
                        })
    return rules





In [4]:
dataset = [
    ["Milk", "Bread", "Butter"],
    ["Beer", "Bread"],
    ["Milk", "Bread", "Beer", "Butter"],
    ["Bread", "Butter"],
    ["Milk", "Bread", "Butter"]
]

In [5]:
tid_list = defaultdict(set)
for tid, transaction in enumerate(dataset):
    for item in transaction:
        tid_list[item].add(tid)

In [6]:
min_support = 2
items = sorted(tid_list.items(), key=lambda x: len(x[1]), reverse=True)
freq_itemsets = eclat([], items, min_support, {})

print("✅ Frequent Itemsets:")
for itemset, support in freq_itemsets.items():
    print(list(itemset), ":", support)

# Generate Rules
rules = generate_rules(freq_itemsets, total_transactions=len(dataset), min_confidence=0.6)

print("\n✅ Association Rules:")
for r in rules:
    print(f"{r['antecedent']} → {r['consequent']} | support={r['support']} | confidence={r['confidence']} | lift={r['lift']}")


✅ Frequent Itemsets:
['Beer'] : 2
['Bread', 'Beer'] : 2
['Milk'] : 3
['Milk', 'Butter'] : 3
['Milk', 'Bread', 'Butter'] : 3
['Milk', 'Bread'] : 3
['Butter'] : 4
['Bread', 'Butter'] : 4
['Bread'] : 5

✅ Association Rules:
['Beer'] → ['Bread'] | support=0.4 | confidence=1.0 | lift=1.0
['Milk'] → ['Butter'] | support=0.6 | confidence=1.0 | lift=1.25
['Butter'] → ['Milk'] | support=0.6 | confidence=0.75 | lift=1.25
['Milk'] → ['Bread', 'Butter'] | support=0.6 | confidence=1.0 | lift=1.25
['Bread'] → ['Milk', 'Butter'] | support=0.6 | confidence=0.6 | lift=1.0
['Butter'] → ['Milk', 'Bread'] | support=0.6 | confidence=0.75 | lift=1.25
['Milk', 'Bread'] → ['Butter'] | support=0.6 | confidence=1.0 | lift=1.25
['Milk', 'Butter'] → ['Bread'] | support=0.6 | confidence=1.0 | lift=1.0
['Bread', 'Butter'] → ['Milk'] | support=0.6 | confidence=0.75 | lift=1.25
['Milk'] → ['Bread'] | support=0.6 | confidence=1.0 | lift=1.0
['Bread'] → ['Milk'] | support=0.6 | confidence=0.6 | lift=1.0
['Bread'] → ['B